# Solvers, Occupations, Spin, k-Points, and Bands

This notebook collects the production-core abstractions added after the toy SCF
prototype:

- dense diagonalization as a tiny-grid reference,
- Davidson-style iterative solving for larger grids,
- fixed and Fermi-Dirac occupations,
- collinear spin-density utilities,
- k-point meshes and non-SCF band diagnostics.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mlx.core as mx
import numpy as np


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not locate repository root")


ROOT = find_repo_root()


## Solver problem statement

At each SCF iteration we need the lowest occupied eigenpairs of

$$
H[\rho]\psi_i = \epsilon_i\psi_i.
$$

Dense diagonalization builds the full Hamiltonian matrix and solves it directly.
That is excellent for tiny reference grids, but the matrix size grows with the
number of grid points \(N_g=N_xN_yN_z\):

$$
H \in \mathbb C^{N_g\times N_g}.
$$

An iterative solver only needs repeated Hamiltonian applications \(H\psi\),
which is the path we eventually want to optimize for Apple Silicon.


## Dense reference versus Davidson-style solver

The dense path is valuable because it is simple to validate.  The iterative path
is the future practical path.  On a tiny grid they should agree closely.


### Residuals and preconditioning

The eigenpair residual is

$$
r_i = H\psi_i-\epsilon_i\psi_i.
$$

A Davidson-style method expands a small subspace using preconditioned residuals.
For a plane-wave/grid Hamiltonian, a simple kinetic preconditioner uses the
dominant kinetic diagonal:

$$
P^{-1}(G)
\approx
\frac{1}{\frac{1}{2}|G|^2-\epsilon_i+\eta}.
$$

This is not the final production solver, but it exposes the diagnostics needed
to judge solver quality: residual norms, subspace size, restart count, and
orthonormality.


In [ ]:
from mlx_atomistic.dft import (
    BandPath,
    DFTSystem,
    EigensolverConfig,
    FermiDiracOccupations,
    FixedOccupations,
    KPointMesh,
    MonkhorstPackGrid,
    SCFConfig,
    magnetization_density,
    run_band_structure,
    run_scf,
    spin_density_from_orbitals,
)

system = DFTSystem.one_center(grid_shape=(4, 4, 4), electron_count=2.0)
dense = run_scf(
    system,
    config=SCFConfig(max_iterations=4, solver="dense", seed=3, convergence_mode="either"),
)
davidson = run_scf(
    system,
    config=SCFConfig(
        max_iterations=4,
        solver="davidson",
        seed=3,
        convergence_mode="either",
        eigensolver_config=EigensolverConfig(max_iterations=6, tolerance=1e-5),
    ),
)

print("dense energy:", dense.total_energy)
print("davidson energy:", davidson.total_energy)
print("solver metadata:", davidson.solver_metadata)


In [ ]:
def residual_trace(result):
    return [row["density_residual"] for row in result.history]

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(residual_trace(dense), marker="o", label="dense")
ax.semilogy(residual_trace(davidson), marker="s", label="davidson")
ax.set_title("SCF residual trace by solver")
ax.set_xlabel("iteration")
ax.set_ylabel("density residual")
ax.legend()
fig.tight_layout()


## Occupations and spin diagnostics

Fractional occupations are needed for metallic or near-degenerate systems.
Collinear spin tracks two densities:

$$
\rho(r)=\rho_\uparrow(r)+\rho_\downarrow(r),
\qquad
m(r)=\rho_\uparrow(r)-\rho_\downarrow(r).
$$


### Fixed versus Fermi-Dirac occupations

For fixed occupations we directly specify \(f_i\).  For finite-temperature
occupations, the electron count is enforced by the chemical potential \(\mu\):

$$
f_i =
\frac{g}{\exp\left((\epsilon_i-\mu)/T_e\right)+1},
\qquad
\sum_i f_i = N_e.
$$

Here \(g=2\) for spin-unpolarized orbitals and \(g=1\) for each collinear spin
channel.  Larger \(T_e\) smooths the occupation step and can make difficult SCF
problems easier, but it also changes the electronic free-energy model.


In [ ]:
eigenvalues = np.array([-0.40, -0.05, 0.02, 0.25], dtype=np.float64)
fixed = FixedOccupations([1.0, 1.0], spin_mode="polarized").resolve()
fermi_low = FermiDiracOccupations(2.0, temperature=0.02).resolve(eigenvalues)
fermi_high = FermiDiracOccupations(2.0, temperature=0.12).resolve(eigenvalues)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(eigenvalues, np.asarray(fermi_low.occupations), marker="o", label="T=0.02")
ax.plot(eigenvalues, np.asarray(fermi_high.occupations), marker="s", label="T=0.12")
ax.set_title("Fermi-Dirac occupations")
ax.set_xlabel("orbital energy / Ha")
ax.set_ylabel("occupation")
ax.legend()
fig.tight_layout()

rho_up, rho_down = spin_density_from_orbitals(
    dense.orbitals,
    dense.orbitals,
    system.grid,
    up_occupations=[1.0],
    down_occupations=[0.6],
)
mag = magnetization_density(rho_up, rho_down)
print("fixed polarized count:", fixed.to_dict())
print("fermi count:", fermi_low.to_dict())
print("magnetization integral:", float(np.asarray(mx.sum(mag) * system.grid.dv)))


## k-point mesh and non-SCF band path

At Γ, the kinetic term is \(0.5|G|^2\).  At a general k point it becomes
\(0.5|G+k|^2\).  Band diagnostics reuse a converged density and do not update
the SCF state.


### Weighted k-point density

For periodic systems the density sums over bands and k-points:

$$
\rho(r)
= \sum_k w_k\sum_n f_{nk}
|\psi_{nk}(r)|^2,
\qquad
\sum_k w_k = 1.
$$

The Γ-point implementation is just the one-point special case.  A band-structure
calculation then freezes the converged density and evaluates eigenvalues along a
path such as \(\Gamma\rightarrow X\).  It should not feed those path states back
into the SCF density.


In [ ]:
gamma = KPointMesh.gamma()
mesh = MonkhorstPackGrid((2, 1, 1))
print("Γ mesh:", gamma.to_dict())
print("2x1x1 mesh:", mesh.to_dict())

path = BandPath.line((0.0, 0.0, 0.0), (0.5, 0.0, 0.0), count=6, start_label="Γ", end_label="X")
bands = run_band_structure(system, dense, path, n_bands=1)
band_values = np.asarray(bands.eigenvalues)[:, 0]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(len(path.points)), band_values, marker="o")
ax.set_title("non-SCF band diagnostic")
ax.set_xlabel("k-path index")
ax.set_ylabel("eigenvalue / Ha")
ax.set_xticks([0, len(path.points) - 1], ["Γ", "X"])
fig.tight_layout()
